# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Praveen23-kk/FlyRank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Lane:** **Lane 2 — Refresh / Content Opportunity Scoring**

**ML Task Type:** **Ranking / Opportunity Scoring** (supported by calibrated probabilistic classification for decline/opportunity risk).

**The One-Paragraph Frame:**
> For **SEO strategists and content editors**, deciding **which existing published articles to review, update, expand, or protect first**, we will build a **ranked opportunity scoring queue** from **observable search visibility (GSC), user engagement (GA4), and content metadata**, predicting **content decline risk (`is_declining_label`)** measured by **Precision@50 and Average Precision (PR-AUC)**. A wrong call costs **wasted editorial writing hours on healthy/growing pages (False Positive)** or **unaddressed traffic and ranking decay on high-value pages (False Negative)**. A plain rule isn't enough because **search performance decay involves complex, non-linear interactions across position drift, CTR-to-position expectations, content age, and engagement quality**. We will claim only **observed, directional, decision-support results** that prioritize human review.

**The Decision & The Action:**
- **Decision:** Given limited weekly editorial capacity (e.g., 20–50 pages reviewed per week), which specific content items should the team audit and refresh first?
- **Action:** Generate an ordered queue where each page receives a priority score and interpretable reason codes (e.g., `declining_with_demand`, `stale_visible_page`, `low_ctr_striking_distance`).
- **Cost of a Wrong Call:** 
  - *False Positive:* Wasting 3–5 hours of editorial and domain-expert writing time rewriting a page that was stable or experiencing normal seasonal variance.
  - *False Negative:* Missing a decaying high-impression page before it slips from Page 1 (rank $\le 10$) into striking distance or deeper, resulting in compounded organic traffic and revenue loss.

In [1]:
import os
import pandas as pd
import numpy as np

# Robust path handling (works from repo root or work/notebooks/)
data_path = "data/raw/content_refresh_anonymized.csv"
if not os.path.exists(data_path):
    data_path = os.path.join("..", "data", "raw", "content_refresh_anonymized.csv")
if not os.path.exists(data_path):
    data_path = os.path.join("..", "..", "data", "raw", "content_refresh_anonymized.csv")

df = pd.read_csv(data_path)

print("=" * 60)
print("FLYRANK STARTER DATASET: EXPLORATORY SLICE SUMMARY")
print("=" * 60)
print(f"Total Rows (Content Items) : {len(df):,}")
print(f"Total Columns               : {df.shape[1]}")
print(f"Pseudonymized Clients       : {df['client_id'].nunique()}")
print(f"Unique Content Items (Grain): {df['content_id'].nunique()}")
print("-" * 60)
print("Breakdown by Content Type:")
print(df['content_type'].value_counts(dropna=False).to_string())
print("-" * 60)
print("Breakdown by 90d Trend Direction (Movement Trajectory):")
trend_counts = df['trend_direction'].value_counts(dropna=False)
trend_pcts = df['trend_direction'].value_counts(normalize=True, dropna=False) * 100
summary_trend = pd.DataFrame({'Count': trend_counts, 'Percentage': trend_pcts.round(2)})
print(summary_trend.to_string())

FLYRANK STARTER DATASET: EXPLORATORY SLICE SUMMARY
Total Rows (Content Items) : 30,000
Total Columns               : 44
Pseudonymized Clients       : 32
Unique Content Items (Grain): 30000
------------------------------------------------------------
Breakdown by Content Type:
content_type
keyword article       27207
feedly article         2096
comparison article      697
------------------------------------------------------------
Breakdown by 90d Trend Direction (Movement Trajectory):
                 Count  Percentage
trend_direction                   
down             16262       54.21
stable            5962       19.87
up                4388       14.63
new               2236        7.45
flat              1152        3.84


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target / Proxy Definition:**
In this starter dataset slice, our target is the binary proxy label:
$$\text{is\_declining\_label} = (\text{trend\_direction} == \text{'down'})$$

**Observed Outcome vs. Defined Rule:**
- **Observed Outcome:** This target is derived from empirical search performance measurements — specifically, whether Google Search Console impressions in the recent 30-day window (`impressions_last_30d`) dropped by more than 20% compared to the prior 30-day window (`impressions_prev_30d`). It reflects **observed real-world traffic loss**, not a subjective human label or internal product decision flag.
- **Proxy Nature & Future Warehouse Target:** While `is_declining_label` in the starter CSV is computed across trailing 30-day sub-windows within the 90-day snapshot, it serves as a starter proxy. In the full ~79M-row daily warehouse release (`fact_content_daily_performance`), this expands to a true forward-looking evaluation design: features calculated over a strictly prior 90-day window ($\text{Day}_{-90} \to \text{Day}_0$) predicting observed decline over a future 30-day window ($\text{Day}_{+1} \to \text{Day}_{+30}$).

**Data Integrity & Leakage Safeguard:**
- Because `is_declining_label` is constructed from `trend_direction` and `trend_pct`, both `trend_direction` and `trend_pct` (as well as direct sub-window columns `impressions_last_30d`, `clicks_last_30d`, `impressions_prev_30d`, etc.) **MUST NEVER be used as model features**.
- Candidate features must strictly contain pre-decision observations (e.g., aggregate 90d activity totals, average position, CTR, engagement rate, content age, word count, and keyword competition).

In [2]:
# Construct the observed decline target
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

pos_count = df['is_declining_label'].sum()
total_count = len(df)
base_rate = (pos_count / total_count) * 100

print("=" * 60)
print("TARGET / PROXY LABEL ANALYSIS: is_declining_label")
print("=" * 60)
print(f"Declining Pages (Label = 1) : {pos_count:,} ({base_rate:.2f}%)")
print(f"Non-Declining Pages (Label = 0): {total_count - pos_count:,} ({100 - base_rate:.2f}%)")
print(f"Dataset Base Rate           : {base_rate:.2f}%")
print("-" * 60)
print("Verification of Excluded Leakage Columns vs Candidate Feature Set:")

leakage_cols = ['trend_direction', 'trend_pct', 'impressions_last_30d', 'impressions_prev_30d', 
                'clicks_last_30d', 'clicks_prev_30d', 'sessions_last_30d', 'sessions_prev_30d']
candidate_feature_sample = ['impressions_90d', 'clicks_90d', 'avg_position', 'ctr', 
                            'sessions_90d', 'engagement_rate', 'scroll_rate', 
                            'content_age_days', 'days_since_last_update', 'word_count']

print(f"Explicitly Excluded Leakage Columns : {leakage_cols}")
print(f"Candidate Safe Feature Subset Sample: {candidate_feature_sample}")
print("Leakage Check: All leakage columns successfully isolated from candidate feature space.")

TARGET / PROXY LABEL ANALYSIS: is_declining_label
Declining Pages (Label = 1) : 16,262 (54.21%)
Non-Declining Pages (Label = 0): 13,738 (45.79%)
Dataset Base Rate           : 54.21%
------------------------------------------------------------
Verification of Excluded Leakage Columns vs Candidate Feature Set:
Explicitly Excluded Leakage Columns : ['trend_direction', 'trend_pct', 'impressions_last_30d', 'impressions_prev_30d', 'clicks_last_30d', 'clicks_prev_30d', 'sessions_last_30d', 'sessions_prev_30d']
Candidate Safe Feature Subset Sample: ['impressions_90d', 'clicks_90d', 'avg_position', 'ctr', 'sessions_90d', 'engagement_rate', 'scroll_rate', 'content_age_days', 'days_since_last_update', 'word_count']
Leakage Check: All leakage columns successfully isolated from candidate feature space.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary Defensible Metric:** **Precision@K (specifically Precision@50 and Precision@20)**

**Why Precision@K Matches the Editorial Decision:**
- Editorial teams have a strict, finite operational capacity: an SEO content team cannot review all 30,000 pages at once. They work through a weekly queue of the **top 20 to 50 surfaced candidates**.
- Standard classification metrics like accuracy are misleading because the base rate is ~54.2%. A naive model predicting all positive gets 54.2% accuracy but provides zero ranking utility.
- **Precision@50** measures the proportion of genuine decay/refresh opportunities within the first 50 recommendations presented to the human reviewer:
$$\text{Precision@K} = \frac{\sum_{i=1}^K \mathbb{I}(y_{\text{rank}(i)} = 1)}{K}$$

**Secondary Metrics:**
- **Average Precision (PR-AUC):** Evaluates ranking quality across all recall levels without threshold dependence.
- **ROC-AUC:** Assesses global pairwise ranking discrimination across the entire catalog.

**What Number Means 'Good'?**
- **Baseline Rule Benchmark:** A standard heuristic baseline (e.g. flagging stale high-impression pages) achieves **$\approx 0.240$ Precision@50** (only ~12 of the top 50 pages are true declining candidates).
- **Target for ML Model:** A learned model (such as Random Forest or Gradient Boosting with proper cross-validation) targeting **Precision@50 $\ge 0.700$** ($\ge 35$ of top 50 pages correctly prioritized) and **ROC-AUC $\ge 0.750$**, representing an actionable $\approx 3\times$ improvement in human reviewer efficiency.

In [3]:
def compute_precision_at_k(y_true, scores, k=50):
    """Compute Precision@K given ground truth labels and ranking scores."""
    ranked_indices = np.argsort(scores)[::-1][:k]
    top_k_labels = np.array(y_true)[ranked_indices]
    return float(np.mean(top_k_labels))

# Baseline heuristic rule for comparison:
# Prioritize by: (High Impressions) * (Content Age / 180) * (Days Since Update / 90)
# A simple heuristic SEO rule often sorts by age and raw impression volume
baseline_heuristic_score = (
    np.log1p(df['impressions_90d']) * 0.4 +
    np.log1p(df['days_since_last_update']) * 0.3 +
    np.log1p(df['content_age_days']) * 0.3
)

p_at_20 = compute_precision_at_k(df['is_declining_label'], baseline_heuristic_score, k=20)
p_at_50 = compute_precision_at_k(df['is_declining_label'], baseline_heuristic_score, k=50)

print("=" * 60)
print("SUCCESS METRIC BENCHMARKING: BASELINE HEURISTIC EVALUATION")
print("=" * 60)
print(f"Overall Dataset Base Rate        : {base_rate:.2f}%")
print(f"Baseline Heuristic Precision@20  : {p_at_20:.3f} ({int(round(p_at_20 * 20))}/20 relevant in top 20)")
print(f"Baseline Heuristic Precision@50  : {p_at_50:.3f} ({int(round(p_at_50 * 50))}/50 relevant in top 50)")
print("-" * 60)
print("Target Bar for ML Model:")
print("  - Target Precision@50 : >= 0.700 (35+/50 relevant)")
print("  - Target ROC-AUC      : >= 0.750")
print("  - Evaluation Goal     : Deliver at least 2.5x - 3x lift in top-K queue precision over naive rules.")

SUCCESS METRIC BENCHMARKING: BASELINE HEURISTIC EVALUATION
Overall Dataset Base Rate        : 54.21%
Baseline Heuristic Precision@20  : 0.500 (10/20 relevant in top 20)
Baseline Heuristic Precision@50  : 0.480 (24/50 relevant in top 50)
------------------------------------------------------------
Target Bar for ML Model:
  - Target Precision@50 : >= 0.700 (35+/50 relevant)
  - Target ROC-AUC      : >= 0.750
  - Evaluation Goal     : Deliver at least 2.5x - 3x lift in top-K queue precision over naive rules.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Definition of the Grain:**
- **One row = One pseudonymized content item (`content_id`) associated with a specific client (`client_id`)**, observed over an aggregated trailing 90-day measurement window.
- **Dataset Scale:** 30,000 unique rows across 32 clients.
- **Identifiers:** `content_id` and `client_id` serve exclusively for grouping, joins, and **client-holdout validation splits** (never as feature inputs).
- **Observable Signal Families per Row:**
  1. *Search Visibility (GSC):* `impressions_90d`, `clicks_90d`, `avg_position`, `ctr`
  2. *User Engagement (GA4):* `sessions_90d`, `pageviews_90d`, `engagement_rate`, `scroll_rate`, `ai_sessions_90d`
  3. *Content & Keyword Metadata:* `content_type`, `main_intent`, `word_count`, `content_age_days`, `days_since_last_update`, `search_volume`, `competition`
  4. *Target:* `is_declining_label`

In [4]:
# Verify grain uniqueness
assert df['content_id'].nunique() == len(df), "Grain violation: content_id is not unique!"

# Select representative columns spanning each signal family
core_columns = [
    'content_id', 'client_id', 'content_type',
    'impressions_90d', 'clicks_90d', 'avg_position', 'ctr',
    'sessions_90d', 'engagement_rate', 'scroll_rate',
    'content_age_days', 'days_since_last_update', 'word_count',
    'is_declining_label'
]

unit_of_analysis_df = df[core_columns].copy()

print("=" * 60)
print("UNIT OF ANALYSIS: ONE ROW = ONE CONTENT ITEM (90-DAY WINDOW)")
print(f"Shape: {unit_of_analysis_df.shape[0]:,} rows x {unit_of_analysis_df.shape[1]} core columns")
print(f"Unique content_id check: {unit_of_analysis_df['content_id'].nunique():,} unique rows (Passed)")
print("=" * 60)

# Display sample of the real dataframe
display_sample = unit_of_analysis_df.head(6)
print(display_sample.to_string(index=False))

UNIT OF ANALYSIS: ONE ROW = ONE CONTENT ITEM (90-DAY WINDOW)
Shape: 30,000 rows x 14 core columns
Unique content_id check: 30,000 unique rows (Passed)
          content_id         client_id    content_type  impressions_90d  clicks_90d  avg_position  ctr  sessions_90d  engagement_rate  scroll_rate  content_age_days  days_since_last_update  word_count  is_declining_label
content_304f48230142 client_f369cb89fc keyword article             3803          29          10.6 0.76            17             5.88         4.55               187                      20      3221.0                   1
content_a1fb4e703a9e client_4e07408562 keyword article            15320           7          20.3 0.05             9             0.00        10.00               445                      25      2481.0                   1
content_9aa793d4d895 client_7f2253d7e2 keyword article            12581          11          36.5 0.09            11             0.00        28.57               141                      

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**1. Tangled, Non-Linear Signal Interactions:**
A simple heuristic (e.g., `if age > 180 and days_since_update > 180 then refresh`) fails because search visibility decay is governed by interdependent forces:
- A page with **high age (300 days)** might be a foundational evergreen guide maintaining Rank #1 with steady CTR and high engagement — flagging it wastes valuable writing resources.
- Conversely, a page with **moderate age (120 days)** at **position 8** with falling CTR relative to its position expectation and poor scroll rate may be rapidly bleeding impressions — but a simple freshness rule ignores it completely.

**2. Brittle Threshold Boundaries:**
- Hardcoded cutoffs create severe boundary artifacts: a page with 499 impressions is ignored while a page with 501 impressions is flagged, even if their underlying trajectory and opportunity scores are identical.
- ML models evaluate continuous gradients of risk, capturing subtle degradation across multiple features rather than triggering on arbitrary binary gates.

**3. Client Heterogeneity and Category Nuance:**
- Different content types (e.g. `keyword article` vs `feedly article`) exhibit distinct baseline missingness patterns and engagement dynamics. A rule-based system requires endless fragile manual if-else branches, whereas tree-based ML naturally splits on feature interactions and segments content types without manual rule maintenance.

In [5]:
# Demonstrate why a typical fixed SEO heuristic rule fails

# Common industry heuristic: Flag if content is stale (>180d), hasn't been updated in >90d, and has high impressions (>=500)
heuristic_flag = (
    (df['content_age_days'] >= 180) & 
    (df['days_since_last_update'] >= 90) & 
    (df['impressions_90d'] >= 500)
)

flagged_total = int(heuristic_flag.sum())
true_positives = int((heuristic_flag & (df['is_declining_label'] == 1)).sum())
false_positives = int((heuristic_flag & (df['is_declining_label'] == 0)).sum())
false_negatives = int((~heuristic_flag & (df['is_declining_label'] == 1)).sum())

heuristic_precision = (true_positives / flagged_total) if flagged_total > 0 else 0
heuristic_recall = true_positives / pos_count

print("=" * 60)
print("EMPIRICAL COMPARISON: FIXED HEURISTIC RULE VS. ACTUAL OUTCOMES")
print("=" * 60)
print(f"Total Pages Flagged by Fixed Rule : {flagged_total:,}")
print(f"True Positives (Correctly Flagged): {true_positives:,}")
print(f"False Positives (Wasted Effort)  : {false_positives:,} ({false_positives/flagged_total*100:.1f}% error rate)")
print(f"False Negatives (Missed Decay)   : {false_negatives:,} ({false_negatives/pos_count*100:.1f}% missed)")
print("-" * 60)
print(f"Fixed Rule Precision             : {heuristic_precision:.3f}")
print(f"Fixed Rule Recall                : {heuristic_recall:.3f}")
print("-" * 60)
print("Analysis of Failure Modes:")
print(f"1. High False Positives: {false_positives:,} stable/growing pages would be needlessly edited.")
print(f"2. Severe False Negatives: {false_negatives:,} decaying pages slipped past the rigid threshold.")
print("Conclusion: A learned ML model captures non-linear interactions across continuous signals,")
print("avoiding brittle threshold cliffs and dramatically increasing queue precision.")

EMPIRICAL COMPARISON: FIXED HEURISTIC RULE VS. ACTUAL OUTCOMES
Total Pages Flagged by Fixed Rule : 4,922
True Positives (Correctly Flagged): 2,855
False Positives (Wasted Effort)  : 2,067 (42.0% error rate)
False Negatives (Missed Decay)   : 13,407 (82.4% missed)
------------------------------------------------------------
Fixed Rule Precision             : 0.580
Fixed Rule Recall                : 0.176
------------------------------------------------------------
Analysis of Failure Modes:
1. High False Positives: 2,067 stable/growing pages would be needlessly edited.
2. Severe False Negatives: 13,407 decaying pages slipped past the rigid threshold.
Conclusion: A learned ML model captures non-linear interactions across continuous signals,
avoiding brittle threshold cliffs and dramatically increasing queue precision.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.